# GENE CRISpy 0.2.0 — Google Colab Quickstart

This notebook installs and exercises the GENE CRISpy Python package from the repository. It demonstrates descriptive FASTA/PDB analysis, the FastAPI endpoint, and the test suite.

> **Scope and privacy:** analysis is descriptive research output, not clinical advice. Do not upload identifiable or sensitive patient/genomic data to a hosted notebook. Colab is a third-party cloud runtime. This quickstart does not configure Firebase, publish the package, or expose a public API tunnel.

## 1. Choose a runtime

In Colab, select **Runtime → Change runtime type → Python 3**. CPU is sufficient for the examples; a GPU is not needed.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/contrivancecompanychicago/replicatedgenecrispy2dpatientworld.git"
REPO_DIR = Path("/content/replicatedgenecrispy2dpatientworld")

if not REPO_DIR.exists():
    !git clone $REPO_URL $REPO_DIR
%cd /content/replicatedgenecrispy2dpatientworld
!git log -1 --oneline

## 2. Install GENE CRISpy and the optional API dependencies

Editable install uses the checked-out repository. The `api` extra installs FastAPI, Uvicorn, and Firebase Admin dependencies; Firebase credentials are not required for the local sequence-analysis endpoint.

In [ ]:
%pip install -e ".[api,test]"

In [ ]:
import genecrispy
from genecrispy import GeneCrispyAgent

print("GENE CRISpy version:", genecrispy.__version__)
agent = GeneCrispyAgent()

## 3. Analyze a sample FASTA sequence

This synthetic example reports sequence type, length, composition, and GC percentage. Replace it only with data you are authorized to process and comfortable running in Colab.

In [ ]:
fasta_text = ">sample_gene\nACGTGC\n"
fasta_result = agent.analyze(fasta_text)
fasta_result

## 4. Analyze a sample PDB structure

The output is a basic coordinate-file summary only; it is not structural validation or molecular simulation.

In [ ]:
pdb_text = (
    "ATOM      1  N   GLY A   1      11.104  13.207   9.223  1.00 20.00           N\n"
    "ATOM      2  CA  GLY A   1      12.000  14.000  10.000  1.00 20.00           C\n"
)
pdb_result = agent.analyze(pdb_text)
pdb_result

## 5. Exercise the FastAPI endpoint locally

Use FastAPI's in-process `TestClient`; this checks the HTTP route without starting a server, exposing a port, or creating a public tunnel.

In [ ]:
from fastapi.testclient import TestClient
from backend import app

client = TestClient(app)
response = client.post(
    "/genecrispy/analyze",
    json={"content": fasta_text, "input_format": "auto"},
)
print("HTTP status:", response.status_code)
response.raise_for_status()
response.json()

## 6. Check invalid-input handling

Malformed or unsupported input should return HTTP 422 with a validation detail.

In [ ]:
bad_response = client.post(
    "/genecrispy/analyze",
    json={"content": ">bad\nACGT?\n", "input_format": "fasta"},
)
print(bad_response.status_code, bad_response.json())
assert bad_response.status_code == 422

## 7. Run the repository tests

This executes the unit and endpoint tests against the checkout.

In [ ]:
!python -m unittest discover -s tests -v

## 8. Optional: build a wheel

This creates a distributable wheel in the Colab runtime's `dist/` folder. It does **not** upload or publish anything. Publishing to PyPI is a separate, deliberate release step.

In [ ]:
%pip install build
!python -m build --wheel
!ls -lh dist

## Notes

- Colab files are temporary unless saved or downloaded; restarting the runtime clears the checkout and build artifacts.
- Do not put GitHub tokens, Firebase service-account keys, or other credentials directly in notebook cells.
- The repository's `/genecrispy/analyze` route is for descriptive FASTA/PDB text analysis. It does not interpret variants or provide clinical guidance.
- To run a server outside this notebook, follow the repository's API guide and use an authenticated, appropriately secured deployment. Do not expose a development server publicly.